In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import math

In [4]:
clinical_data = pd.read_csv("datasets/clinical_data.csv")
lab_data1 = pd.read_csv("datasets/lab_data.csv")

Filtering data for 30 days

In [5]:
preop = lab_data1[(lab_data1['dt'] < 0) & (lab_data1['dt'] >= -2592000)] #30 days

preop_last = (preop.sort_values('dt').groupby(['caseid', 'name']).last()   
              .reset_index()[['caseid','dt', 'name', 'result']])

preop_last.head(10)
print("pre-op rows:", len(preop))

pre-op rows: 196862


In [6]:
preop.groupby("name")["dt"].describe()
print("median hours before surgery:", (preop_last["dt"].abs()/3600).median())

median hours before surgery: 29.634444444444444


In [7]:
lab_data = (preop_last
            .pivot(index="caseid", columns="name", values="result")
            .reset_index())
lab_data.columns.name = None

lab_cols = [c for c in lab_data.columns if c != "caseid"]
lab_data[lab_cols] = lab_data[lab_cols].apply(pd.to_numeric, errors="coerce")

lab_data.head(10)
print(lab_data.shape)
print(clinical_data.shape)

(5388, 35)
(6388, 74)


filtering columns that have more than 75 missing data

In [8]:
missing_rate = lab_data.isna().mean()
lab_data = lab_data.loc[:, missing_rate < 0.75]
print("after 75% filter:", lab_data.shape)
lab_data


after 75% filter: (5388, 24)


,caseid,alb,alt,aptt,ast,bun,cl,cr,crp,fib,...,k,na,p,plt,pt%,ptinr,ptsec,tbil,tprot,wbc
0,1,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,...,3.5,141.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN,115.0,NaN,NaN,NaN,...,4.7,143.0,NaN,251.0,NaN,NaN,NaN,NaN,NaN,4.36
2,4,NaN,NaN,NaN,NaN,NaN,108.0,NaN,NaN,NaN,...,4.3,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,4.6,30.0,36.5,64.0,27.0,108.0,1.83,NaN,178.0,...,4.9,139.0,5.4,158.0,81.0,1.14,13.0,0.9,8.2,14.64
4,6,3.9,17.0,29.1,19.0,14.0,104.0,1.07,0.01,183.0,...,3.4,141.0,2.8,145.0,104.0,0.97,11.1,1.0,6.7,3.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5383,6384,NaN,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,...,4.2,143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5384,6385,3.7,28.0,NaN,18.0,19.0,109.0,0.84,NaN,NaN,...,4.0,144.0,NaN,239.0,NaN,NaN,NaN,0.6,5.9,5.45
5385,6386,NaN,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,...,4.5,142.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5386,6387,4.4,10.0,32.2,16.0,7.0,103.0,0.65,0.03,239.0,...,3.6,142.0,3.0,214.0,90.0,1.07,12.2,0.3,6.9,7.70


Remove physiologically impossible values only

In [9]:
lab_data = lab_data[lab_data["hct"].between(10, 65) | lab_data["hct"].isna()]
for c in ["height", "weight", "bmi"]:
    clinical_data.loc[clinical_data[c] <= 0, c] = np.nan
print("lab_data after plausibility filter:", lab_data.shape)
print("clinical_data after plausibility filter:", clinical_data.shape)

lab_data after plausibility filter: (5388, 24)
clinical_data after plausibility filter: (6388, 74)


Fix the syntax error in data

In [10]:
clinical_data["age"] = clinical_data["age"].replace(">89", 90)
clinical_data["age"] = pd.to_numeric(clinical_data["age"], errors="coerce")

Merge 2 dataframes with common IDs

In [11]:
import pandas as pd
from scipy.stats import pearsonr

merged_data = clinical_data.merge(lab_data, on="caseid", how="inner")

In [12]:
import pandas as pd
from scipy.stats import pearsonr

merged_data = clinical_data.merge(lab_data, on="caseid", how="inner")
merged_data["sex_bin"] = (merged_data["sex"] == "M").astype(int)

predictors = ["sex_bin", "age", "height", "weight", "bmi"]

results = []
for col in predictors:
    sub = merged_data[[col, "hct"]].dropna()   # pairwise deletion, like .corr()
    r, p = pearsonr(sub[col], sub["hct"])
    results.append({"variable": col, "pearson_r": r, "p_value": p, "n": len(sub)})

corr_df = pd.DataFrame(results)
print(corr_df)

  variable  pearson_r        p_value     n
0  sex_bin   0.241466   5.681913e-56  4138
1      age  -0.061301   7.950962e-05  4138
2   height   0.257384   1.323553e-63  4138
3   weight   0.365879  2.846304e-131  4138
4      bmi   0.265882   6.616834e-68  4138


This is the dataframe on which we will build our ML models

In [13]:
merged_data = merged_data.dropna(subset=['hct','sex_bin','height','weight','bmi'])
merged_data


,caseid,subjectid,casestart,caseend,anestart,aneend,opstart,opend,adm,dis,...,na,p,plt,pt%,ptinr,ptsec,tbil,tprot,wbc,sex_bin
1,2,2487,0,15741,-1039,14921.000000,1721,14621,-221160,1506840,...,143.0,NaN,251.0,NaN,NaN,NaN,NaN,NaN,4.36,1
3,5,4416,0,21531,-1009,22391.000000,2591,20291,-67560,3734040,...,139.0,5.4,158.0,81.0,1.14,13.0,0.9,8.2,14.64,1
4,6,5586,0,5145,-51,4749.000000,2649,4449,-300600,131400,...,141.0,2.8,145.0,104.0,0.97,11.1,1.0,6.7,3.20,0
5,7,5124,0,15770,477,14817.000000,3177,14577,-154320,623280,...,139.0,NaN,229.0,109.0,0.95,10.8,0.4,7.4,4.28,0
8,11,3963,0,5265,-222,5117.999999,618,4518,-135540,37260,...,141.0,NaN,261.0,121.0,0.90,10.2,0.3,5.8,6.31,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5381,6382,5436,0,6876,-136,6764.000000,764,5864,-227820,204180,...,141.0,3.2,323.0,111.0,0.94,10.7,0.9,6.9,5.22,0
5382,6383,4255,0,23242,-1883,23137.000000,3037,22641,-224040,812760,...,141.0,4.8,202.0,114.0,0.92,10.5,0.6,6.5,5.01,1
5384,6385,2278,0,20643,-544,20996.000000,2396,19496,-225600,1675200,...,144.0,NaN,239.0,NaN,NaN,NaN,0.6,5.9,5.45,1
5386,6387,5230,0,12025,-550,12830.000000,1730,11030,-227760,377040,...,142.0,3.0,214.0,90.0,1.07,12.2,0.3,6.9,7.70,0
